# PhotoPrism MLOps Infrastructure — Provisioning

This notebook provisions the infrastructure for the PhotoPrism MLOps project on Chameleon Cloud (KVM@TACC).

**What it creates:**
- 3 VM instances (node1, node2, node3) on KVM@TACC
- A private network (192.168.1.0/24) connecting all nodes
- Ports on sharednet1 with security groups (SSH, HTTP, service ports)
- A floating IP on node1 (the only public-facing node)

**Prerequisites:**
- Chameleon account with access to KVM@TACC
- `clouds.yaml` with valid application credentials uploaded to `/work/clouds.yaml`
- SSH key `id_rsa_chameleon` registered on Chameleon

**After provisioning, from your local terminal:**
1. Copy SSH key: `scp -i ~/.ssh/id_rsa_chameleon ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>:~/.ssh/id_rsa_chameleon`
2. SSH to node1: `ssh -i ~/.ssh/id_rsa_chameleon cc@<FLOATING_IP>`
3. Run setup: `git clone https://github.com/akashchauhanweb/photoprism-mlops-infra.git && bash photoprism-mlops-infra/scripts/setup_cluster.sh`

## Step 1: Configure Chameleon Context

In [ ]:
from chi import server, context, lease, network
import chi, os, time, datetime, subprocess, shutil, json

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

# ---- Project configuration ----
PROJECT_PREFIX = "proj24"
SSH_KEY_NAME   = "id_rsa_chameleon"
LEASE_END      = datetime.datetime(2026, 4, 27, 3, 59, tzinfo=datetime.timezone.utc)
VM_FLAVOR      = "m1.xlarge"
VM_COUNT       = 3
VM_IMAGE       = "CC-Ubuntu24.04"

## Step 2: Install Terraform

In [ ]:
TF_VERSION = "1.14.4"

commands = [
    "mkdir -p /work/.local/bin",
    f"wget -q https://releases.hashicorp.com/terraform/{TF_VERSION}/terraform_{TF_VERSION}_linux_amd64.zip",
    f"unzip -o -q terraform_{TF_VERSION}_linux_amd64.zip",
    "mv terraform /work/.local/bin",
    f"rm terraform_{TF_VERSION}_linux_amd64.zip",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {cmd}\n{result.stderr}")
    else:
        print(f"Done: {cmd}")

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]

result = subprocess.run("terraform --version", shell=True, capture_output=True, text=True)
print(result.stdout.split('\n')[0])

## Step 3: Create Lease

In [ ]:
LEASE_NAME = f"lease-infra-{PROJECT_PREFIX}"

l = lease.Lease(
    LEASE_NAME,
    end_date=LEASE_END,
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id(VM_FLAVOR),
    amount=VM_COUNT,
)
l.submit(idempotent=True)

reservation_id = l.get_reserved_flavors()[0].id
print(f"Lease: {LEASE_NAME} — Status: ACTIVE")
print(f"Reservation flavor ID: {reservation_id}")

## Step 4: Set Up Terraform Files

Creates the Terraform configuration adapted from the course lab IaC repo (`gourmetgram-iac`).

**Resources created by Terraform:**
- Private network + subnet (192.168.1.0/24, no gateway)
- 3 ports on private network (fixed IPs, no port security)
- 3 ports on sharednet1 (with security groups)
- 3 compute instances (CC-Ubuntu24.04, m1.medium)
- 1 floating IP assigned to node1

In [ ]:
# Cell: Clone infra repo and set up Terraform
import subprocess, shutil

INFRA_REPO = "https://github.com/akashchauhanweb/photoprism-mlops-infra.git"
infra_dir = "/work/photoprism-mlops-infra"
tf_dir = f"{infra_dir}/tf/kvm"

# Clone or pull the repo
if os.path.exists(infra_dir):
    result = subprocess.run("git pull", shell=True, cwd=infra_dir, capture_output=True, text=True)
    print(f"Repo updated: {result.stdout.strip()}")
else:
    result = subprocess.run(f"git clone {INFRA_REPO} {infra_dir}", shell=True, capture_output=True, text=True)
    print(f"Repo cloned: {result.stdout.strip()}")

# Copy clouds.yaml into tf directory (secret — not in repo)
shutil.copy("/work/clouds.yaml", f"{tf_dir}/clouds.yaml")
print(f"clouds.yaml copied to {tf_dir}")

# Verify Terraform files exist
import os
tf_files = [f for f in os.listdir(tf_dir) if f.endswith('.tf')]
print(f"Terraform files found: {', '.join(sorted(tf_files))}")

## Step 5: Terraform Init, Plan, and Apply

In [ ]:
# Build a clean environment — remove Chameleon Jupyter's OS_ vars
# which conflict with our clouds.yaml (they point to CHI@UC)
clean_env = {k: v for k, v in os.environ.items() if not k.startswith("OS_")}
clean_env["OS_CLOUD"]        = "openstack"
clean_env["PATH"]            = "/work/.local/bin:" + clean_env.get("PATH", "")
clean_env["HOME"]            = os.environ.get("HOME", "/home/jovyan")
clean_env["TF_VAR_suffix"]   = PROJECT_PREFIX
clean_env["TF_VAR_key"]      = SSH_KEY_NAME
clean_env["TF_VAR_reservation"] = reservation_id

def run_tf(command, description):
    print(f"\n{'='*60}")
    print(f"  {description}")
    print(f"{'='*60}")
    result = subprocess.run(
        command, shell=True, cwd=tf_dir, env=clean_env,
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise Exception(f"{description} failed with return code {result.returncode}")
    return result.returncode

run_tf("terraform init", "Terraform Init")
run_tf("terraform validate", "Terraform Validate")
run_tf("terraform plan", "Terraform Plan")

In [ ]:
# Apply — creates all resources
run_tf("terraform apply -auto-approve", "Terraform Apply")

# Extract outputs
result = subprocess.run(
    "terraform output -json",
    shell=True, cwd=tf_dir, env=clean_env,
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)
floating_ip = outputs["floating_ip"]["value"]

print(f"\n{'='*60}")
print(f"  Infrastructure provisioned successfully!")
print(f"{'='*60}")
print(f"\n  Floating IP: {floating_ip}")
print(f"\n  Next steps from your local terminal:")
print(f"  1. scp -i ~/.ssh/id_rsa_chameleon ~/.ssh/id_rsa_chameleon ~/.ssh/sealed-secrets-key-backup.yaml cc@{floating_ip}:~/.ssh/")
print(f"  2. ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print(f"  3. git clone https://github.com/akashchauhanweb/photoprism-mlops-infra.git && bash photoprism-mlops-infra/scripts/setup_cluster.sh")

---

## Teardown

Run these cells **only** when you want to destroy all infrastructure and free resources.

In [ ]:
# TEARDOWN: Destroy all Terraform-managed resources
# run_tf("terraform destroy -auto-approve", "Terraform Destroy")

In [ ]:
# TEARDOWN: Delete the lease
# l.delete()
# print("Lease deleted.")